# Resume / Continue Training from Checkpoint

Two modes:
- **Resume**: restore full state (model + optimizer + scheduler + RNG + epoch counter) and continue the same run.
- **Continue**: load model weights only, fresh optimizer/scheduler, optionally with a different config (regularizer, LR, epochs, etc.).

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath('..')
DATA_DIR = os.path.join(REPO_ROOT, 'data')
PRETRAIN = os.path.join(REPO_ROOT, 'experiments', 'pretrain')
SAVE_DIR = os.path.join(REPO_ROOT, 'runs', 'lejepa_v1')

sys.path.insert(0, PRETRAIN)
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))
sys.path.insert(0, REPO_ROOT)

import torch, time
import matplotlib.pyplot as plt
from pathlib import Path

from configs import Config, DATASET_INFO
from data import get_dataloaders, InMemoryGPUDataset
from models import LeJEPAEncoder, LinearProbe
from losses import build_regularizer, build_sigreg
from scheduler import make_scheduler
from trainer import setup_seed, load_for_continuation
from checkpoint import save_checkpoint, load_checkpoint
from train_loops import (
    train_epoch_standard_inmem,
    train_epoch_standard_loader,
    train_epoch_pooled_inmem,
    train_epoch_pooled_loader,
    make_nograd_loader,
    evaluate,
    eval_distribution,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Save dir: {SAVE_DIR}')

In [ ]:
# List available checkpoints
for run_dir in sorted(Path(SAVE_DIR).iterdir()):
    if not run_dir.is_dir():
        continue
    ckpts = sorted(run_dir.glob('*.pt'))
    if ckpts:
        names = [c.name for c in ckpts]
        print(f'{run_dir.name}/  ->  {names}')

## Configure checkpoint + training

In [ ]:
def run_tag(cfg):
    method = cfg.regularizer
    if cfg.accumulate:
        method += '_pooled'
    return f'{cfg.dataset}_{cfg.encoder_scale}_{method}_bs{cfg.batch_size}_seed{cfg.seed}'


def run_training_from_checkpoint(checkpoint_path, mode, cfg, save_dir=SAVE_DIR):
    """Load a checkpoint and continue training.

    mode="resume":   restore full state, continue in the same run directory.
    mode="continue": load encoder+probe weights only, fresh optimizer/scheduler,
                     save to a new cont_* directory.
    """
    assert mode in ('resume', 'continue')
    assert os.path.exists(checkpoint_path), f'Checkpoint not found: {checkpoint_path}'

    ckpt_data = torch.load(checkpoint_path, map_location=device, weights_only=False)
    ckpt_cfg = ckpt_data['config']

    # ── Build config ──
    if mode == 'resume':
        # Restore original config, override epochs to allow extending
        cfg = Config(**{k: v for k, v in ckpt_cfg.items() if hasattr(Config, k)})
        cfg.epochs = max(cfg.epochs, ckpt_data['epoch'] + 1 + 100)  # at least 100 more

    # ── Run directory ──
    tag = run_tag(cfg)
    if mode == 'resume':
        run_dir = Path(checkpoint_path).parent
    else:
        base_tag = Path(checkpoint_path).parent.name
        run_dir = Path(save_dir) / f'cont_{base_tag}__{tag}'
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f'Run dir: {run_dir}')

    # ── Data + model ──
    setup_seed(cfg.seed)
    train_source, val_source, gpu_aug = get_dataloaders(cfg, device)
    in_memory = isinstance(train_source, InMemoryGPUDataset)

    # ── Optimizer + scheduler ──
    encoder = LeJEPAEncoder(cfg).to(device)
    probe_dim = encoder.hidden_dim if cfg.probe_on_emb else cfg.proj_dim
    probe = LinearProbe(probe_dim, cfg.num_classes).to(device)
    if cfg.use_compile:
        encoder = torch.compile(encoder)
    print(f'{tag}  |  params={sum(p.numel() for p in encoder.parameters()):,}')

    enc_opt = torch.optim.AdamW(encoder.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay, betas=(0.99, 0.999))
    probe_opt = torch.optim.AdamW(probe.parameters(), lr=cfg.probe_lr, weight_decay=cfg.probe_wd)

    if in_memory:
        steps_per_epoch = len(train_source) // cfg.batch_size
    else:
        steps_per_epoch = len(train_source)
    warmup_steps = cfg.warmup_epochs * steps_per_epoch

    enc_sched = make_scheduler(enc_opt, warmup_steps, cfg.lr)
    probe_sched = make_scheduler(probe_opt, warmup_steps, cfg.probe_lr)

    # ── Load checkpoint ──
    start_epoch = 0
    global_step = 0
    best_val_acc = 0.0

    if mode == 'resume':
        start_epoch, global_step, best_val_acc = load_checkpoint(
            checkpoint_path, encoder, probe, enc_opt, probe_opt,
            enc_sched, probe_sched, cfg)
        print(f'Resumed from epoch {start_epoch}, step {global_step}, '
              f'best_acc {best_val_acc:.4f}')
    else:
        load_for_continuation(checkpoint_path, encoder, probe, device)
        print(f'Loaded encoder+probe from {checkpoint_path}')

    # ── Regularizer ──
    reg_fn = None
    sigreg_mod = None
    nograd_loader = None
    nograd_iter_state = [None]
    fifo = FIFOBuffer(cfg.fifo_size) if cfg.fifo_size > 0 else None

    if not cfg.accumulate:
        reg_fn = build_regularizer(cfg, device)
    else:
        sigreg_mod = build_sigreg(cfg, device)
        if not in_memory and cfg.nograd_pool_size > 0:
            nograd_bs = cfg.nograd_pool_size
            nograd_loader = make_nograd_loader(
                train_source.dataset, nograd_bs, cfg.num_workers)
            nograd_iter_state = [iter(nograd_loader)]

    sample_gen = torch.Generator(device=device).manual_seed(cfg.seed) if in_memory else None
    history = {'epoch': [], 'train_loss': [], 'val_acc': [],
               'lam': [], 'mu': [], 'eps': [],
               'w1': [], 'w2': [], 'mean_norm': [], 'cov_frob_rel': [],
               'cov_diag_mean': [], 'cov_offdiag_max': [],
               'proj_eff_rank': [], 'emb_eff_rank': [],
               'emb_eff_rank_ratio': [], 'emb_dim': [],
               'emb_w1': [], 'emb_w2': [], 'emb_cov_frob_rel': []}
    global_step = 0
    best_val_acc = 0.0

    # ── Training loop ──
    for epoch in range(start_epoch, cfg.epochs):
        t0 = time.time()
        if cfg.accumulate and in_memory:
            avg_loss, global_step = train_epoch_pooled_inmem(
                epoch, encoder, probe, train_source,
                gpu_aug, enc_opt, probe_opt, enc_sched, probe_sched, cfg,
                global_step, sample_gen, sigreg_mod,
                fifo=fifo)
        elif cfg.accumulate:
            avg_loss, global_step = train_epoch_pooled_loader(
                epoch, encoder, probe, train_source,
                gpu_aug, nograd_loader, nograd_iter_state,
                enc_opt, probe_opt, enc_sched, probe_sched, cfg,
                global_step, sigreg_mod, fifo=fifo)
        elif in_memory:
            avg_loss, global_step = train_epoch_standard_inmem(
                epoch, encoder, probe, reg_fn, train_source,
                gpu_aug, enc_opt, probe_opt, enc_sched, probe_sched,
                cfg, global_step, sample_gen)
        else:
            avg_loss, global_step = train_epoch_standard_loader(
                epoch, encoder, probe, reg_fn, train_source,
                gpu_aug, enc_opt, probe_opt, enc_sched, probe_sched,
                cfg, global_step)

        val_acc = evaluate(encoder, probe, val_source, cfg)
        dist = eval_distribution(encoder, val_source, cfg)
        elapsed = time.time() - t0

        history['epoch'].append(epoch)
        history['train_loss'].append(avg_loss)
        history['val_acc'].append(val_acc)
        for k in ('w1', 'w2', 'mean_norm', 'cov_frob_rel',
                  'cov_diag_mean', 'cov_offdiag_max',
                  'proj_eff_rank', 'emb_eff_rank',
                  'emb_eff_rank_ratio', 'emb_dim',
                  'emb_w1', 'emb_w2', 'emb_cov_frob_rel'):
            history[k].append(dist.get(k, float('nan')))

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        if (epoch + 1) % cfg.eval_interval == 0:
            tag = 'emb' if cfg.probe_on_emb else 'proj'
            print(f'  epoch {epoch} | loss={avg_loss:.4f} val({tag})={val_acc:.4f} '
                  f'best={best_val_acc:.4f} | '
                  f'w1={dist["w1"]:.3f} emb_w1={dist.get("emb_w1", 0):.3f} | '
                  f'rank proj={dist["proj_eff_rank"]:.2f}/{cfg.proj_dim} '
                  f'emb={dist["emb_eff_rank"]:.2f}/{dist["emb_dim"]} | '
                  f'{elapsed:.1f}s')

        if (epoch + 1) % cfg.save_interval == 0:
            save_checkpoint(
                str(run_dir / f'epoch{epoch+1}.pt'), encoder, probe,
                enc_opt, probe_opt, enc_sched, probe_sched,
                epoch, global_step, cfg, best_val_acc)

    final_path = run_dir / 'final.pt'
    save_checkpoint(
        str(final_path), encoder, probe,
        enc_opt, probe_opt, enc_sched, probe_sched,
        cfg.epochs - 1, global_step, cfg, best_val_acc)
    print(f'Saved: {final_path}  (best val_acc={best_val_acc:.4f})')
    return history

In [ ]:
# ── Pick a checkpoint ──
CHECKPOINT_PATH = os.path.join(
    SAVE_DIR, 'cifar100_resnet18_sigreg_bs128_seed42', 'final.pt')

# "resume" = full state restore (same run, same config)
# "continue" = load weights only, fresh optimizer, new config below
MODE = "continue"

# STL10 (5000 train @ 96px) is the fast iteration setup — should run
# at ~3-4s/epoch vs ~37s for CIFAR100. Switch DATASET back to 'cifar100'
DATASET = "cifar100"#'flowers102'
BS = 128
EPOCHS = 1000
reg = 'sigreg'
# reg = 'w1'
pooled = False

cfg = Config(
    dataset=DATASET, data_dir=DATA_DIR,
    encoder_scale='resnet18',
    regularizer=reg, accumulate=pooled,
    batch_size=BS, epochs=EPOCHS, num_inv_nograd_views=0, detach_inv_centroid=False,
    use_compile=True, num_workers=4, 
    eval_interval=1, save_interval=25,
    log_interval=100, seed=42, nograd_pool_size=8, fifo_size=0,
    proj_dim=128, inv_on_emb=False, probe_on_emb=True,
    num_aug_views=8, lr=5e-4, 
    blur_p=0.75, solarize_p=0.75,
    lambd = 0.05, lambd_emb = 0.0, crop_scale=(0.05, 1.0)
)

history = run_training_from_checkpoint(CHECKPOINT_PATH, MODE, cfg)

In [ ]:
if history is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.plot(history['epoch'], history['val_acc'], linewidth=2)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Val Accuracy')
    ax1.set_title('Validation Accuracy'); ax1.grid(True, alpha=0.3)

    ax2.plot(history['epoch'], history['train_loss'], linewidth=2)
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Train Loss')
    ax2.set_title('Training Loss'); ax2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    # Distributional metrics
    fig2, axd = plt.subplots(1, 4, figsize=(20, 4.5))
    axd[0].plot(history['epoch'], history['w1'], linewidth=2)
    axd[0].set_xlabel('Epoch'); axd[0].set_ylabel('sliced W1 to N(0,1)')
    axd[0].set_title('Proj W1')

    axd[1].plot(history['epoch'], history.get('emb_w1', []), linewidth=2)
    axd[1].set_xlabel('Epoch'); axd[1].set_ylabel('sliced W1 to N(0,1)')
    axd[1].set_title('Emb W1')

    axd[2].plot(history['epoch'], history['cov_frob_rel'], linewidth=2)
    axd[2].set_xlabel('Epoch'); axd[2].set_ylabel('||cov - I||_F / ||I||_F')
    axd[2].set_title('Covariance distance to I (proj)')

    axd[3].plot(history['epoch'], history['mean_norm'], linewidth=2)
    axd[3].set_xlabel('Epoch'); axd[3].set_ylabel('||mean(proj)||_2')
    axd[3].set_title('Mean magnitude (proj)')
    plt.tight_layout(); plt.show()

    # Rank diagnostics
    fig3, axr = plt.subplots(1, 2, figsize=(12, 4.5))
    axr[0].plot(history['epoch'], history['proj_eff_rank'], linewidth=2)
    axr[0].set_xlabel('Epoch'); axr[0].set_ylabel('effective rank (proj)')
    axr[0].set_title('Projector eff. rank')

    axr[1].plot(history['epoch'], history['emb_eff_rank'], linewidth=2)
    emb_dim_seen = history['emb_dim'][-1] if history['emb_dim'] else None
    if emb_dim_seen is not None:
        axr[1].axhline(emb_dim_seen, color='black', linestyle=':', alpha=0.4,
                       label=f'hidden_dim={emb_dim_seen}')
        axr[1].legend()
    axr[1].set_xlabel('Epoch'); axr[1].set_ylabel('effective rank (emb)')
    axr[1].set_title('Backbone eff. rank')
    plt.tight_layout(); plt.show()